# Sistemi lineari e problema dei minimi quadrati

## Sistemi lineari

Consideriamo un sistema lineare quadrato

$$
A\mathbf{x}=\mathbf{b},
\qquad
A\in\mathbb{R}^{n\times n},
\quad\mathbf{x},\mathbf{b}\in\mathbb{R}^{n}.
$$

Il vettore $\mathbf{b}$ è noto, mentre $\mathbf{x}$ è il vettore delle incognite.

Lo studio numerico dei sistemi lineari comprende tre questioni fondamentali:

1. **Esistenza e unicità:** il sistema possiede una soluzione? La soluzione è unica?
2. **Algoritmi:** come possiamo calcolare la soluzione in modo efficiente?
3. **Errore:** quanto è affidabile la soluzione calcolata in aritmetica finita?

## Esistenza e unicità della soluzione

Se $A$ è quadrata, le seguenti proprietà sono equivalenti:

- $A$ è non singolare;
- $\det(A)\neq0$;
- $\operatorname{rank}(A)=n$;
- $\ker(A)=\{\mathbf{0}\}$;
- esiste $A^{-1}$;
- per ogni $\mathbf{b}\in\mathbb{R}^n$, il sistema $A\mathbf{x}=\mathbf{b}$ ammette un'unica soluzione.

In questo caso, dal punto di vista teorico,

$$
\mathbf{x}=A^{-1}\mathbf{b}.
$$

Se $A$ è singolare, il sistema può non avere soluzioni oppure può averne infinite, a seconda del vettore $\mathbf{b}$.

## Perché non si calcola l'inversa?

La formula $\mathbf{x}=A^{-1}\mathbf{b}$ è utile dal punto di vista teorico, ma normalmente **non viene utilizzata come algoritmo**.

Calcolare esplicitamente $A^{-1}$:

- richiede più operazioni della soluzione diretta del sistema;
- richiede memoria per memorizzare l'intera matrice inversa;
- può introdurre errori di arrotondamento non necessari;
- calcola molte informazioni che non servono se desideriamo soltanto $\mathbf{x}$.

Nella pratica si usa quindi un algoritmo che risolve direttamente

$$
A\mathbf{x}=\mathbf{b}
$$

senza costruire $A^{-1}$.

## Metodi diretti e metodi iterativi


I **metodi diretti** producono la soluzione dopo un numero finito di operazioni, se si lavora in aritmetica esatta. Sono basati principalmente sull'eliminazione di Gauss e sulle fattorizzazioni della matrice.

I **metodi iterativi** costruiscono invece una successione di approssimazioni

$$
\mathbf{x}^{(0)},\mathbf{x}^{(1)},\mathbf{x}^{(2)},\ldots
$$

che, sotto opportune condizioni, converge alla soluzione. Sono particolarmente importanti per matrici grandi e sparse.

Ora considereremo i **metodi diretti**, nelle ultime lezioni vedremo invece un metodo iterativo.

## Matrici particolari: sistemi triangolari

Una matrice $L=(\ell_{ij})$ è **triangolare inferiore** se

$$
\ell_{ij}=0\qquad\text{per }j>i.
$$

Una matrice $U=(u_{ij})$ è **triangolare superiore** se

$$
u_{ij}=0\qquad\text{per }i>j.
$$

I sistemi triangolari sono semplici da risolvere perché le incognite possono essere calcolate una alla volta.



Consideriamo il sistema triangolare inferiore

$$
L\mathbf{x}=\mathbf{b}.
$$

La prima equazione contiene soltanto $x_1$. Una volta calcolato $x_1$, la seconda equazione permette di determinare $x_2$, e così via.

La formula generale è

$$
x_i=\frac{1}{\ell_{ii}}
\left(b_i-\sum_{j=1}^{i-1}\ell_{ij}x_j\right),
\qquad i=1,\ldots,n.
$$

Il metodo prende il nome di **sostituzione in avanti**. Richiede un numero di operazioni proporzionale a $n^2$.


Consideriamo ora il sistema triangolare superiore

$$
U\mathbf{x}=\mathbf{b}.
$$

Si parte dall'ultima equazione e si procede verso la prima:

$$
x_i=\frac{1}{u_{ii}}
\left(b_i-\sum_{j=i+1}^{n}u_{ij}x_j\right),
\qquad i=n,n-1,\ldots,1.
$$

Questo procedimento è detto **sostituzione all'indietro** e ha anch'esso complessità $\mathcal{O}(n^2)$.

In [7]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)

def sostituzione_avanti(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n)

    for i in range(n):
        if abs(L[i, i]) < 1e-14:
            raise ValueError('Elemento diagonale nullo')
        x[i] = (b[i] - L[i, :i] @ x[:i]) / L[i, i]
    return x


def sostituzione_indietro(U, b):
    U = np.asarray(U, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        if abs(U[i, i]) < 1e-14:
            raise ValueError('Elemento diagonale nullo')
        x[i] = (b[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x

## Costo computazionale

Per un sistema triangolare di ordine $n$, il numero complessivo di termini presenti nelle somme è

$$
1+2+\cdots+(n-1)=\frac{n(n-1)}{2}.
$$

Una sostituzione in avanti o all'indietro richiede quindi circa $n^2$ operazioni aritmetiche, cioè

$$
\mathcal{O}(n^2).
$$

Questo è molto meno del costo $\mathcal{O}(n^3)$ necessario per fattorizzare una matrice densa generale.

## Risoluzione del sistema mediante LU

Consideriamo ora il sistema 

$$
A\mathbf{x}=\mathbf{b}
$$

in cui la matrice $A$ è la precedente:

$$
A=
\begin{pmatrix}
2&1&1\\
4&-6&0\\
-2&7&2
\end{pmatrix}.
$$
 e il termine noto è: 

$$
\mathbf{b}=
\begin{pmatrix}
5\\-2\\9
\end{pmatrix}.
$$

Dalla fattorizzazione $A=LU$ abbiamo ottenuto le seguenti matrici:

$$
U=
\begin{pmatrix}
2&1&1\\
0&-8&-2\\
0&0&1
\end{pmatrix}.
$$

$$
L=
\begin{pmatrix}
1&0&0\\
2&1&0\\
-1&-1&1
\end{pmatrix}.
$$

### Primo passaggio: $L\mathbf{y}=\mathbf{b}$

La sostituzione in avanti fornisce

$$
\mathbf{y}=
\begin{pmatrix}
5\\-12\\2
\end{pmatrix}.
$$

### Secondo passaggio: $U\mathbf{x}=\mathbf{y}$

La sostituzione all'indietro fornisce

$$
\mathbf{x}=
\begin{pmatrix}
1\\1\\2
\end{pmatrix}.
$$

## Risoluzione di un sistema con fattorizzazione Lu con pivoting

Quando la matrice $A$ è fattorizzata con pivoting, si ha che:

$PA=LU$ dove $P$ è una matrice di permutazione.

Allora il sistema lineare $Ax=b$ è equivalente al sistema: $PAx=Pb$, cioè $LUx=Pb$.

Si risolvono quindi in sequenza:
1. $Ly=Pb$
2. $Ux=y$.


## Risoluzione del sistema con pivoting

Scegliamo

$$
\mathbf{b}=
\begin{pmatrix}
3\\0\\7
\end{pmatrix}.
$$

Poiché $PA=LU$, dobbiamo prima permutare il termine noto:

$$
P\mathbf{b}=
\begin{pmatrix}
7\\0\\3
\end{pmatrix}.
$$

La sostituzione in avanti nel sistema

$$
L\mathbf{y}=P\mathbf{b}
$$

fornisce

$$
\mathbf{y}=
\begin{pmatrix}
7\\-\frac72\\1
\end{pmatrix}.
$$

Risolvendo poi $U\mathbf{x}=\mathbf{y}$ otteniamo

$$
\mathbf{x}=
\begin{pmatrix}
1\\2\\-1
\end{pmatrix}.
$$

In [ ]:
b_piv = np.array([3., 0., 7.])

y_piv = sostituzione_avanti(L_piv, P @ b_piv)
x_piv = sostituzione_indietro(U_piv, y_piv)

print('P @ b =', P @ b_piv)
print('y =', y_piv)
print('x =', x_piv)
print('Residuo b - Ax =', b_piv - A_piv @ x_piv)
print('Norma del residuo =', np.linalg.norm(b_piv - A_piv @ x_piv))

P @ b = [7. 0. 3.]
y = [ 7.  -3.5  1. ]
x = [ 1.  2. -1.]
Residuo b - Ax = [0. 0. 0.]
Norma del residuo = 0.0


## Risoluzione di un sistema con fattorizzazione di Cholesky 

Se $A$ è una matrice simmetrica e definita positiva, abbiamo visti che si può fattorizzare tramite l'algoritmo di Cholesky come: $A=LL^T$.

Allora  il sistema

$$
A\mathbf{x}=\mathbf{b}
$$

diventa

$$
LL^T\mathbf{x}=\mathbf{b}.
$$

Si risolvono quindi due sistemi triangolari:

$$
L\mathbf{y}=\mathbf{b},
\qquad
L^T\mathbf{x}=\mathbf{y}.
$$



In [8]:
b = np.array([6., 11., -1.])

# np.linalg.solve applicato ai due sistemi triangolari
y = np.linalg.solve(L, b)
x = np.linalg.solve(L.T, y)

print('y =', y)
print('x =', x)
print('Norma del residuo =', np.linalg.norm(b - A @ x))

NameError: name 'L' is not defined

## Esercizi

1. Verificare manualmente che le matrici $L$ e $U$ dell'esempio soddisfino $LU=A$.
2. Risolvere lo stesso sistema con `np.linalg.solve` e confrontare il risultato.
3. Cambiare il termine noto $\mathbf{b}$ senza ricalcolare $L$ e $U$.
4. Contare moltiplicazioni e divisioni effettuate dalle due funzioni di sostituzione.
5. Provare la funzione `fattorizzazione_lu` sulla matrice

   $$
   \begin{pmatrix}
   0&1\\1&1
   \end{pmatrix}
   $$

   e spiegare perché è necessario uno scambio di righe.